# Library Download

In [2]:
%pip install torch torch-geometric
%pip install ogb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


# GCN

## Library, Data load

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.utils import degree
from torch_geometric.loader import DataLoader
from ogb.graphproppred import PygGraphPropPredDataset, Evaluator

dataset = PygGraphPropPredDataset(name="ogbg-code2")
split_idx = dataset.get_idx_split()

train_loader = DataLoader(dataset[split_idx["train"]], batch_size=32, shuffle=True)
val_loader   = DataLoader(dataset[split_idx["valid"]], batch_size=32)
test_loader  = DataLoader(dataset[split_idx["test"]],  batch_size=32)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

Using device: cuda


## Utils

In [4]:
def get_vocab_mapping(seq_list, num_vocab):
    vocab_cnt, vocab_list = {}, []
    for seq in seq_list:
        for w in seq:
            if w in vocab_cnt: vocab_cnt[w] += 1
            else: vocab_cnt[w] = 1; vocab_list.append(w)

    cnt_list = np.array([vocab_cnt[w] for w in vocab_list])
    topvocab = np.argsort(-cnt_list, kind='stable')[:num_vocab]
    print(f'Vocab coverage: {float(np.sum(cnt_list[topvocab]))/np.sum(cnt_list):.4f}')

    vocab2idx = {vocab_list[i]: idx for idx, i in enumerate(topvocab)}
    idx2vocab = [vocab_list[i] for i in topvocab]
    vocab2idx['__UNK__'] = num_vocab;   idx2vocab.append('__UNK__')
    vocab2idx['__EOS__'] = num_vocab+1; idx2vocab.append('__EOS__')
    return vocab2idx, idx2vocab

In [5]:
def encode_y_to_arr(data, vocab2idx, max_seq_len):
    seq = data.y
    augmented = seq[:max_seq_len] + ['__EOS__'] * max(0, max_seq_len - len(seq))
    data.y_arr = torch.tensor(
        [[vocab2idx.get(w, vocab2idx['__UNK__']) for w in augmented]], dtype=torch.long
    )
    return data

def decode_arr_to_seq(arr, idx2vocab):
    eos_pos = torch.nonzero(arr == len(idx2vocab)-1, as_tuple=False)
    arr = arr[:torch.min(eos_pos).item()] if len(eos_pos) > 0 else arr
    return [idx2vocab[i.item()] for i in arr]

In [6]:
def augment_edge(data):
    ei = data.edge_index
    ea = torch.zeros(ei.size(1), 2)
    ei_inv = torch.stack([ei[1], ei[0]], dim=0)
    ea_inv = torch.cat([torch.zeros(ei_inv.size(1),1), torch.ones(ei_inv.size(1),1)], dim=1)

    attr_nodes = torch.where(data.node_is_attributed.view(-1) == 1)[0]
    if len(attr_nodes) > 1:
        ei_next = torch.stack([attr_nodes[:-1], attr_nodes[1:]], dim=0)
        ea_next = torch.cat([torch.ones(ei_next.size(1),1), torch.zeros(ei_next.size(1),1)], dim=1)
        ei_next_inv = torch.stack([ei_next[1], ei_next[0]], dim=0)
        ea_next_inv = torch.ones(ei_next.size(1), 2)
        data.edge_index = torch.cat([ei, ei_inv, ei_next, ei_next_inv], dim=1)
        data.edge_attr  = torch.cat([ea, ea_inv, ea_next, ea_next_inv], dim=0)
    else:
        data.edge_index = torch.cat([ei, ei_inv], dim=1)
        data.edge_attr  = torch.cat([ea, ea_inv], dim=0)
    return data

## Node Encode

In [7]:
import torch

class ASTNodeEncoder(torch.nn.Module):
    """
    x[:, 0] : node type index
    x[:, 1] : node attribute index
    depth    : depth of the node in AST (별도 텐서)
    """
    def __init__(self, emb_dim, num_nodetypes, num_nodeattributes, max_depth):
        super().__init__()
        self.max_depth = max_depth
        self.type_encoder      = torch.nn.Embedding(num_nodetypes,      emb_dim)
        self.attribute_encoder = torch.nn.Embedding(num_nodeattributes, emb_dim)
        self.depth_encoder     = torch.nn.Embedding(max_depth + 1,      emb_dim)

    def forward(self, x, depth):
        depth = depth.clone()
        depth[depth > self.max_depth] = self.max_depth
        return self.type_encoder(x[:, 0]) + self.attribute_encoder(x[:, 1]) + self.depth_encoder(depth)

## Model

In [8]:
class GCNConv(MessagePassing):
    def __init__(self, emb_dim):
        super().__init__(aggr='add')
        self.linear       = nn.Linear(emb_dim, emb_dim)
        self.root_emb     = nn.Embedding(1, emb_dim)
        self.edge_encoder = nn.Linear(2, emb_dim)

    def forward(self, x, edge_index, edge_attr):
        x        = self.linear(x)
        edge_emb = self.edge_encoder(edge_attr.float())
        row, _   = edge_index
        deg          = degree(row, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5).clamp(max=1e9)
        norm = deg_inv_sqrt[edge_index[0]] * deg_inv_sqrt[edge_index[1]]
        agg  = self.propagate(edge_index, x=x, edge_attr=edge_emb, norm=norm)
        return agg + F.relu(x + self.root_emb.weight) / deg.view(-1,1).clamp(min=1)

    def message(self, x_j, edge_attr, norm):
        return norm.view(-1,1) * F.relu(x_j + edge_attr)
    
class GCN(nn.Module):
    def __init__(self, num_vocab, max_seq_len, node_encoder, num_layer=5, emb_dim=300, drop_ratio=0.5):
        super().__init__()
        self.convs      = nn.ModuleList([GCNConv(emb_dim) for _ in range(num_layer)])
        self.bns        = nn.ModuleList([nn.BatchNorm1d(emb_dim) for _ in range(num_layer)])
        self.node_encoder = node_encoder
        self.drop_ratio = drop_ratio
        self.num_layer  = num_layer

        self.pred_heads = nn.ModuleList([
            nn.Linear(emb_dim, num_vocab) for _ in range(max_seq_len)
        ])
        self.max_seq_len = max_seq_len

    def forward(self, data):
        h = self.node_encoder(data.x, data.node_depth.view(-1))
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            h = bn(conv(h, data.edge_index, data.edge_attr))
            h = F.dropout(F.relu(h) if i < self.num_layer-1 else h,
                          p=self.drop_ratio, training=self.training)
        h_graph = global_mean_pool(h, data.batch)
        return [head(h_graph) for head in self.pred_heads]

## train

In [9]:
NUM_VOCAB, MAX_SEQ_LEN = 5000, 5
BATCH_SIZE, LR, EPOCHS = 32, 1e-3, 30

dataset   = PygGraphPropPredDataset(name='ogbg-code2')
split_idx = dataset.get_idx_split()

vocab2idx, idx2vocab = get_vocab_mapping([dataset[i].y for i in split_idx['train']], NUM_VOCAB)

def transform_fn(data):
    return encode_y_to_arr(augment_edge(data), vocab2idx, MAX_SEQ_LEN)

dataset.transform = transform_fn
train_loader = DataLoader(dataset[split_idx['train']], batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(dataset[split_idx['valid']], batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(dataset[split_idx['test']],  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

num_nodetypes      = int(dataset.data.x[:, 0].max().item()) + 1
num_nodeattributes = int(dataset.data.x[:, 1].max().item()) + 1
print(f'num_nodetypes: {num_nodetypes}, num_nodeattributes: {num_nodeattributes}')

node_encoder = ASTNodeEncoder(300, num_nodetypes, num_nodeattributes, max_depth=20)

model     = GCN(len(idx2vocab), MAX_SEQ_LEN, node_encoder).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
evaluator = Evaluator('ogbg-code2')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Vocab coverage: 0.9026


/src/gs25103/.local/lib/python3.8/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
/src/gs25103/.local/lib/python3.8/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to supp

num_nodetypes: 97, num_nodeattributes: 10030
Parameters: 11,032,910


In [ ]:
from tqdm import tqdm

def train(loader):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc='Training', leave=False)
    for data in pbar:
        data = data.to(DEVICE)
        optimizer.zero_grad()
        label = data.y_arr.to(torch.long)
        loss  = sum(criterion(pred, label[:,i]) for i, pred in enumerate(model(data)))
        loss.backward(); optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    refs, preds = [], []
    for data in loader:
        data     = data.to(DEVICE)
        pred_arr = torch.stack([p.argmax(-1) for p in model(data)], dim=1).cpu()
        for i in range(pred_arr.size(0)):
            preds.append(decode_arr_to_seq(pred_arr[i],        idx2vocab))
            refs.append( decode_arr_to_seq(data.y_arr[i].cpu(), idx2vocab))
    return evaluator.eval({'seq_ref': refs, 'seq_pred': preds})['F1']

best = 0.0
EPOCHS = 3
for epoch in range(1, EPOCHS+1):
    loss = train(train_loader)
    f1   = evaluate(val_loader)
    if f1 > best:
        best = f1; torch.save(model.state_dict(), 'best_model.pt')
    print(f'Epoch {epoch:03d} | Loss: {loss:.4f} | Val F1: {f1:.4f} | Best: {best:.4f}')

model.load_state_dict(torch.load('best_model.pt'))
print(f'Test F1: {evaluate(test_loader):.4f}')

1


Epoch 001 | Loss: 14.5891 | Val F1: 0.1827 | Best: 0.1827
2


Epoch 002 | Loss: 13.6512 | Val F1: 0.1886 | Best: 0.1886
3


Epoch 003 | Loss: 13.1289 | Val F1: 0.1960 | Best: 0.1960
Test F1: 0.1787
